In [ ]:
# Importar as bibliotecas

import datetime
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
import sys
import matplotlib.pyplot as plt

In [ ]:
# Resolver os 'imports' do projeto 

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from utils.classes.pandas_dataframe import PandasDataframe as pd_dataframe

In [ ]:
# Trazer as opções de configuração do JSON

json_file = os.path.abspath('../../options.json')

with open(json_file, 'r', encoding='utf-8') as j_file:
    json_data = json.load(j_file)

In [ ]:
# Declaração dos caminhos

dir_archive = os.path.abspath('../../data')

dir_cvm     = os.path.join(dir_archive, 'cvm')
dir_metrics = os.path.join(dir_archive, 'metrics')

dir_cvm_processed = os.path.join(dir_cvm, json_data['DIR']['CVM']['DATA_NAME'])

In [ ]:
# Manejo da data dos investimentos

START_DATE = json_data['CHARTS']['START_DATE']
END_DATE   = json_data["CHARTS"]['END_DATE']

In [ ]:
# Principais funções para costrução dos gráficos

def compile_investments_in_df (investment_list: list[str]) -> list[pd.DataFrame]:
    """
    Using a list of CNPJ investments, its searched each archive investment by its name and compiled in a dateframe.

    Parameters:
        investment_list (list[str]): A list that contains CNPJ investments.
    
    Returns:
        investment_archives (list[pd.Dateframes]): A list that contains dataframes of each investment.
    """
    CSV_EXTENSION = '.csv'

    investment_archives = []
    for investment_cnpj in investment_list:
        investment_archive_name = json_data['INVESTMENTS'][investment_cnpj]['INVESTMENT_FUND']
        investment_archive_name = investment_archive_name.replace(' ', '_')

        investment_archive_path = os.path.join(dir_cvm, json_data['DIR']['CVM']['DATA_NAME'], investment_archive_name + CSV_EXTENSION)
        investment_archive = pd_dataframe(investment_archive_path, None)
        investment_archive.csv_to_df() 
    
        investment_archives.append(investment_archive)

    return investment_archives

def compile_metrics_in_df (metric_list: list[str]) -> list[pd.DataFrame]:
    """
    Using a list of metrics names, its searched each archive metric by its name and compiled using in a dateframe.

    Parameters:
        metric_list (list[str]): A list that contains metrics names.
    
    Returns:
        metrics_archives (list[pd.Dateframes]): A list that contains dataframes of each metric.
    """
    CSV_EXTENSION = '.csv'

    metrics_archives = []
    for metric_key_name in metric_list:
        metric_archive_path = os.path.join(dir_metrics, metric_key_name.lower(), metric_key_name.lower() + '_valuation' + CSV_EXTENSION)
        metric_archive = pd_dataframe(metric_archive_path, None)
        metric_archive.csv_to_df()

        metrics_archives.append(metric_archive)

    return metrics_archives

def warn_invalid_dates (investment_archives: list[pd.DataFrame], metrics_archives: list[pd.DataFrame]) -> None:
    """
    All the start dates need to be in the same date.

    Parameters:
        investment_archives (list[pd.Dataframe]): A list with investment dataframes.
        metrics_archives (list[pd.Dataframe]): A list with metrics dataframes.
    
    Returns:
        None  
    """
    reference_investment = {}

    for investment_archive in investment_archives:

        if reference_investment.get('DT_COMPTC', '') != '':
            investment_row = investment_archive.find_row_data(0)

            if investment_row['DT_COMPTC'] != reference_investment['DT_COMPTC']:
                print("The investment row " + investment_row['NOME_FUNDOS'] + "has a difference date compared with investment reference " + investment_row['NOME_FUNDOS'] + ".")
                print("Investment Row (" + investment_row['NOME_FUNDOS'] + "): " + investment_row['DT_COMPTC'])
                print("Investment Reference (" + reference_investment['NOME_FUNDOS'] + "): " + reference_investment['DT_COMPTC'])
        else:
            reference_investment = investment_archive.find_row_data(0)

    for metric_name in metrics_archives:
        
        metric_row = metric_name.find_row_data(0)

        if metric_row['DATE'] != reference_investment['DT_COMPTC']:
            print("The investment row " + investment_row['NOME_FUNDOS'] + "has a difference date compared with investment reference " + investment_row['NOME_FUNDOS'] + ".")
            print("Investment Row (" + investment_row['NOME_FUNDOS'] + "): " + investment_row['DT_COMPTC'])
            print("Metric Reference (" + metric_row['METRIC_NAME'] + "): " + metric_row['DATE'])

def flateness_list (data_lists: list[list[str]]) -> list[str]:
    """
    Turn a list of lists into a singular list.

    Parameters:
        data_lists (list[list[str]]): A list with lists.
    
    Returns:
        metrics_archives (list[pd.Dateframes]): A flatness list with the others lists.
    """
    flat_list = []
    for data_list in data_lists:
        for data in data_list:
            flat_list.append(data)
    return flat_list

def compile_in_dataframe (cvm_archives: list[pd.DataFrame], metric_archives: list[pd.DataFrame]) -> pd.DataFrame:
    """
    Compile columns ('Name', 'Valuation Percent' and 'Date') from investments and metrics into a singular dataframe.

    Parameters:
        cvm_archives (list[pd.DataFrame]): A list with lists.
        metric_archives (list[pd.DataFrame])
    
    Returns:
        chart_df (list[pd.Dateframes]): Data compiled to be used on charts.
    """

    [name_data, valuation_data, date_data] = [[], [], []]

    for cvm_archive in cvm_archives:

        cvm_name = cvm_archive.get_column_in_list('NOME_FUNDOS')
        name_data.append(cvm_name)

        cvm_valuation = cvm_archive.get_column_in_list('VALUATION_PERCENT')
        valuation_data.append(cvm_valuation)

        cvm_date = cvm_archive.get_column_in_list('DT_COMPTC')
        date_data.append(cvm_date)

    for metric_archive in metric_archives:

        metric_name = metric_archive.get_column_in_list('METRIC_NAME')
        name_data.append(metric_name)

        metric_valuation = metric_archive.get_column_in_list('VALUATION_PERCENT')
        valuation_data.append(metric_valuation)

        metric_date = metric_archive.get_column_in_list('DATE')
        date_data.append(metric_date)

    name_data      = flateness_list(name_data)
    valuation_data = flateness_list(valuation_data)
    date_data      = flateness_list(date_data)

    object_chart_data = {
        'NAME': name_data,
        'VALUATION_PERCENT': valuation_data,
        'DATE': date_data,
    }

    chart_df = pd_dataframe(None, None, dict=object_chart_data)
    chart_df.dict_to_df()

    return chart_df

def order_by_date (chart_data: pd.DataFrame) -> pd.DataFrame:
    """
    Order the chart data by 'DATE' without desynchronized the columns.

    Parameters:
        chart_data (pd.DataFrame): A list with lists.
    
    Returns:
        chart_data (pd.Dateframes]: Data compiled to be used on charts.
    """
    chart_data.df['DATE'] = pd.to_datetime(chart_data.df['DATE'])
    chart_data.sort_elements_list(['DATE'])
    return chart_data

In [ ]:
# Gráfico de valorização

investment_cnpj_list = json_data['CHARTS']['INVESTMENT_LIST']
metric_names_list    = json_data['CHARTS']['METRICS_LIST']

investment_archives = compile_investments_in_df (investment_cnpj_list)
metrics_archives    = compile_metrics_in_df (metric_names_list)
warn_invalid_dates(investment_archives, metrics_archives)

chart_data = compile_in_dataframe(investment_archives, metrics_archives)
chart_data = order_by_date(chart_data)

sns.set_style("whitegrid")
plt.figure(figsize=(20, 6))

plot = sns.lineplot(
    data=chart_data.df, 
    x='DATE', 
    y='VALUATION_PERCENT', 
    hue='NAME', 
    linewidth=2.5
)

plt.title("Evolução do percentual de valorização", fontsize=16, fontweight='bold', pad=20)
plt.text(
    1.225, 0.0,
    "Atualizado em " + str(datetime.datetime.today().date()), 
    transform=plt.gca().transAxes, 
    fontsize=10, 
    va='bottom', ha='left'
)

plt.xlabel("Período", fontsize=14)
plt.ylabel("Percentual de valorização (%)", fontsize=14)
plt.xticks(
    rotation=45, 
    fontsize=11,
    ha='right',
    rotation_mode= 'anchor'
)
plt.legend(
    title='Fundo de Investimento', 
    bbox_to_anchor=(1.05, 1), 
    fontsize=11
)

plt.gca().tick_params(axis='x', pad=5)
plt.show()